# Week 3 Quiz — Pipelines, Performance & Streaming

**Topics:** Batch Pipelines, Spark UI, Memory Management, Pandas API on Spark, Structured Streaming, Auto Loader, DLT, Workflows

**15 questions.** Run each answer cell to reveal the correct answer and explanation.

---

## Q1

A data engineer notices a Spark job is running slowly. They want to see detailed metrics broken down **per executor** — including GC time, shuffle read/write, and task failures per executor. Which tool provides this view?

- A. The Spark driver logs (stdout)
- B. The Ganglia UI
- **C. The Spark UI Executors tab**
- D. `dbutils.fs.ls("/logs")`

In [ ]:
answer = "C"
explanation = (
    "The Spark UI Executors tab provides per-executor metrics: memory used, GC time, task count, "
    "shuffle read/write bytes, and failure counts. This is where you identify if one executor is "
    "overloaded (skew) or has excessive GC. Driver logs show driver-side exceptions only. "
    "Ganglia monitors OS-level hardware (CPU, RAM, network) not Spark-specific task metrics. "
    "dbutils.fs.ls lists DBFS paths, not Spark execution metadata."
)
print(f"Q1 → {answer}  |  {explanation}")

## Q2

After enabling Adaptive Query Execution (AQE), a data engineer wants to control how many shuffle partitions Spark creates after a shuffle stage. Which Spark configuration property governs this?

- **A. `spark.sql.shuffle.partitions`**
- B. `spark.executor.cores`
- C. `spark.default.parallelism`
- D. `spark.sql.files.maxPartitionBytes`

In [ ]:
answer = "A"
explanation = (
    "spark.sql.shuffle.partitions sets the target number of post-shuffle partitions for "
    "DataFrame/SQL operations (default: 200). With AQE enabled, Spark uses this as the "
    "initial target and then coalesces smaller partitions at runtime. "
    "spark.default.parallelism controls RDD operations, not DataFrame shuffles. "
    "spark.executor.cores sets CPU allocation per executor. "
    "spark.sql.files.maxPartitionBytes controls how large a partition can be when reading files."
)
print(f"Q2 → {answer}  |  {explanation}")

## Q3

Which of the following statements about Structured Streaming DataFrames is **correct**?

- A. Streaming DataFrames support all batch DataFrame operations including `show()` and `count()`
- **B. Structured Streaming uses an unbounded table model where each new micro-batch of data is appended to a conceptually infinite table**
- C. Streaming DataFrames cannot be joined with static (batch) DataFrames
- D. Structured Streaming always requires the `append` output mode

In [ ]:
answer = "B"
explanation = (
    "The core Structured Streaming abstraction is the 'unbounded input table' — conceptually "
    "an infinite table where each new arrival appends a row. Queries over this table produce "
    "a 'result table' that is written to a sink with each trigger. "
    "NOT all batch operations work on streaming DFs — show() and count() require a sink. "
    "Stream-static joins ARE supported. Output mode (append/complete/update) depends on the "
    "query type, not always append."
)
print(f"Q3 → {answer}  |  {explanation}")

## Q4

A data engineer inspects a Spark ETL job that consistently hits performance bottlenecks. The Spark UI shows that most time is spent on **task time (CPU computation)**. The team reports the cluster is low on compute resources. Which action should the engineer take?

- A. Repartition data and spread jobs across more cluster nodes to fix under-utilization
- B. No change needed — high CPU task time indicates efficient cluster usage
- C. Increase parallelism to resolve over-utilized memory
- **D. Tune executor and core configuration or resize the cluster to provide more CPU capacity**

In [ ]:
answer = "D"
explanation = (
    "High task (CPU) time means the cluster is compute-bound — tasks are spending most of their "
    "duration doing actual computation rather than waiting on I/O, shuffle, or scheduling. "
    "This aligns with the team's report of being 'low on compute resources'. The fix is to "
    "add more CPU capacity: add executors, add cores per executor, or scale up the cluster. "
    "Option A is for under-utilization (opposite problem). Option C conflates CPU with memory."
)
print(f"Q4 → {answer}  |  {explanation}")

## Q5 (Multi-select — choose 2)

A Spark ETL job fails with `java.lang.OutOfMemoryError: Java heap space`. Which **two** recovery actions should the data engineer perform?

- **A. Narrow the filters in the query to reduce the amount of data collected**
- **B. Upsize the worker nodes and activate adaptive shuffle partitions (AQE)**
- C. Upsize the driver node and deactivate adaptive shuffle partitions
- D. Cache the dataset to boost query performance
- E. Fix shuffle partitions to 50

In [ ]:
answer = "A and B"
explanation = (
    "A: Narrowing filters reduces data volume processed by each task, directly reducing memory pressure. "
    "B: Larger worker nodes provide more heap per executor; AQE dynamically coalesces small partitions "
    "and splits skewed ones, preventing any single task from holding excessive data in memory. "
    "C: The error is on workers (heap), not the driver — upsizing the driver won't help; "
    "disabling AQE would worsen the memory situation. "
    "D: Caching consumes additional memory and could exacerbate OOM. "
    "E: Fixing to 50 partitions could create very large partitions, worsening OOM."
)
print(f"Q5 → {answer}  |  {explanation}")

## Q6

A data engineer is debugging a Python notebook in Databricks. The notebook fails with an error during a DataFrame transformation. The engineer wants to **set breakpoints and inspect variable values** (including DataFrames) in real-time. Which tool should they use?

- A. Databricks CLI to download and analyze driver logs
- **B. The Python notebook interactive Debugger to set breakpoints and inspect variables**
- C. The Ganglia UI to monitor cluster resource usage
- D. The Spark UI to analyze the execution plan

In [ ]:
answer = "B"
explanation = (
    "Databricks notebooks have a built-in Python Debugger (based on ipdb/pdb) that lets engineers "
    "set breakpoints, step through code line by line, and inspect variable values including "
    "DataFrames at runtime. This is the direct tool for interactive notebook debugging. "
    "The Databricks CLI manages workspace operations, not code debugging. "
    "Ganglia monitors cluster hardware metrics, not Python code execution. "
    "Spark UI shows execution plans but cannot inspect Python variables at breakpoints."
)
print(f"Q6 → {answer}  |  {explanation}")

## Q7

Which of the following queries performs a **streaming hop from raw data to a bronze table** in the medallion architecture?

- A. `spark.table("sales").withColumn("avgPrice", col("sales")/col("units")).writeStream.option("checkpointLocation", path).outputMode("append").table("newSales")`
- **B. `spark.readStream.load(rawSalesLocation).writeStream.option("checkpointLocation", path).outputMode("append").table("newSales")`**
- C. `spark.read.load(rawSalesLocation).write.mode("append").table("newSales")`
- D. `spark.table("sales").groupBy("store").agg(sum("sales")).writeStream.option("checkpointLocation", path).outputMode("complete").table("newSales")`

In [ ]:
answer = "B"
explanation = (
    "A raw-to-bronze hop reads from a raw landing location (not an existing Delta table) and "
    "writes to a new table using streaming (readStream + writeStream) so it processes data "
    "incrementally. Option B does exactly this: reads raw data as a stream and writes it to "
    "'newSales' with a checkpoint. "
    "Option A reads from an existing Delta table ('sales') — that's a silver-level hop. "
    "Option C is batch (spark.read, not readStream). "
    "Option D performs aggregation — that's a gold-level hop."
)
print(f"Q7 → {answer}  |  {explanation}")

## Q8

A data engineer needs to identify which files in a shared cloud directory are **new since the previous pipeline run** and ingest only those files. The existing files must not be moved or deleted. Which tool solves this problem?

- A. Databricks SQL
- B. Delta Lake
- C. Unity Catalog
- D. Data Explorer
- **E. Auto Loader**

In [ ]:
answer = "E"
explanation = (
    "Auto Loader (spark.readStream.format('cloudFiles')) tracks which files have been processed "
    "using checkpointing and either directory listing or cloud file notification APIs. On each run "
    "it processes only newly arrived files without reprocessing existing ones and without modifying "
    "the source directory. Delta Lake provides ACID storage but does not handle source file tracking. "
    "Databricks SQL, Unity Catalog, and Data Explorer have no incremental file-ingestion capabilities."
)
print(f"Q8 → {answer}  |  {explanation}")

## Q9

For Structured Streaming to reliably track its exact progress and recover from failures, Spark records the **offset range of data processed in each trigger** using which two mechanisms?

- **A. Checkpointing and Write-ahead Logs (WALs)**
- B. Checkpointing and Idempotent Sinks
- C. Structured Streaming cannot track offset ranges
- D. Replayable Sources and Idempotent Sinks
- E. Write-ahead Logs and Idempotent Sinks

In [ ]:
answer = "A"
explanation = (
    "Structured Streaming achieves fault tolerance through two internal mechanisms: "
    "(1) Checkpointing — persists query progress metadata (offsets, state) to reliable storage "
    "so the query can resume from exactly where it stopped after a failure. "
    "(2) Write-ahead Logs (WALs) — record the offset range of each trigger BEFORE processing, "
    "so on restart Spark knows which data to re-read. "
    "Idempotent sinks and replayable sources are end-to-end exactly-once guarantees, "
    "not the mechanism Spark uses to record offsets internally."
)
print(f"Q9 → {answer}  |  {explanation}")

## Q10

A data engineer wants a streaming query to execute a **micro-batch every 5 seconds**. Which line of code should be added to the `writeStream` chain?

- A. `.trigger("5 seconds")`
- B. `.trigger()`
- C. `.trigger(once="5 seconds")`
- **D. `.trigger(processingTime="5 seconds")`**
- E. `.trigger(continuous="5 seconds")`

In [ ]:
answer = "D"
explanation = (
    "trigger(processingTime='5 seconds') sets a fixed-interval micro-batch trigger. Spark waits "
    "at least 5 seconds between micro-batches (or longer if the batch takes more than 5s). "
    "trigger() with no args runs as fast as possible. "
    "trigger(once=True) runs one micro-batch then stops. "
    "trigger(availableNow=True) processes all current data in one or more micro-batches then stops. "
    "trigger(continuous='1 second') enables continuous processing — a different execution engine, "
    "not a micro-batch with a 5-second interval."
)
print(f"Q10 → {answer}  |  {explanation}")

## Q11

A data engineer has a multi-task Databricks job where one task **fails intermittently 10% of the time**. Which action ensures the job completes each night while **minimizing compute costs**?

- A. Institute a retry policy for the entire job
- B. Observe the task manually each run to determine the cause
- C. Schedule the job to run multiple times per night so at least one succeeds
- **D. Institute a retry policy specifically for the task that periodically fails**
- E. Use a jobs cluster for each task in the job

In [ ]:
answer = "D"
explanation = (
    "Databricks Jobs supports task-level retry policies. Setting retries only on the failing task "
    "means the job automatically retries that specific task on failure without rerunning the other "
    "tasks that succeeded. This is the most compute-efficient approach. "
    "Option A retries the whole job — all tasks re-run even those that passed, wasting compute. "
    "Option B requires manual intervention, breaking automation. "
    "Option C runs all tasks multiple times, the most expensive approach. "
    "Option E doesn't address the retry problem."
)
print(f"Q11 → {answer}  |  {explanation}")

## Q12

A data engineer is processing a streaming table with Delta Live Tables (DLT). They need to filter out rows where `order_datetime IS NULL` from `orders_raw` and store the result in `orders_valid`. Which DLT code snippet is correct?

- **A. `CREATE OR REFRESH STREAMING TABLE orders_valid CONSTRAINT valid_data EXPECT (order_datetime IS NOT NULL) ON VIOLATION DROP ROW AS SELECT * FROM orders_raw;`**
- B. Same as A but with an additional `WHERE order_datetime IS NOT NULL` clause
- C. `CREATE OR REFRESH STREAMING TABLE orders_valid AS SELECT * FROM orders_raw WHERE order_datetime IS NOT NULL;`
- D. `CREATE OR REPLACE STREAMING TABLE orders_valid FILTER (order_datetime IS NOT NULL) AS SELECT * FROM orders_raw;`

In [ ]:
answer = "A"
explanation = (
    "In Delta Live Tables, the CONSTRAINT ... EXPECT ... ON VIOLATION DROP ROW pattern is the "
    "idiomatic way to enforce data quality and filter rows. It names the rule ('valid_data'), "
    "defines the expectation, and specifies the action (drop violating rows). This also records "
    "metrics about dropped rows in the DLT pipeline event log. "
    "Option B redundantly adds a WHERE clause that duplicates what ON VIOLATION DROP ROW already does. "
    "Option C works but bypasses DLT's quality monitoring framework. "
    "Option D uses invalid syntax (FILTER clause, CREATE OR REPLACE instead of CREATE OR REFRESH)."
)
print(f"Q12 → {answer}  |  {explanation}")

## Q13

A data engineer needs to ingest streaming data from a **Kafka topic** called `orders_topic` into Databricks using Structured Streaming. Which code snippet is correct?

- A. `df.orders_json().load("kafka://orders_topic")`
- **B. `spark.readStream.format("kafka").option("kafka.bootstrap.servers","kafka_broker").option("subscribe","orders_topic").load()`**
- C. `df.read.kafka("orders_topic")`
- D. `spark.readStream.format("file").load("/kafka/orders_topic")`
- E. `df.kafka().option("topic","orders_topic").load("kafka_broker")`

In [ ]:
answer = "B"
explanation = (
    "Reading from Kafka in Structured Streaming requires: "
    "spark.readStream.format('kafka') to specify the Kafka source, "
    ".option('kafka.bootstrap.servers','host:port') to connect to brokers, and "
    ".option('subscribe','orders_topic') to name the topic. "
    "Option A is fabricated syntax (no .orders_json() method). "
    "Option C uses batch read and a non-existent .kafka() method. "
    "Option D uses 'file' format for a Kafka topic, which is meaningless. "
    "Option E is completely invented API syntax."
)
print(f"Q13 → {answer}  |  {explanation}")

## Q14

A data engineer uses Auto Loader to ingest data from cloud storage. After a schema change in the source, they want the pipeline to **halt and fail** until someone reviews and confirms the new schema is intentional. Which Auto Loader schema evolution mode should they set?

- **A. `failOnNewColumns`**
- B. `none`
- C. `rescue`
- D. `addNewColumns`

In [ ]:
answer = "A"
explanation = (
    "failOnNewColumns causes Auto Loader to throw an exception and stop the stream when new "
    "columns are detected that aren't in the current schema. The pipeline stays down until a "
    "data engineer reviews the change and explicitly updates the schema. "
    "'none' silently ignores new columns. "
    "'rescue' captures mismatched data in a _rescued_data column without failing. "
    "'addNewColumns' automatically extends the schema — the opposite of what's needed here."
)
print(f"Q14 → {answer}  |  {explanation}")

## Q15

A data engineer deploys a multi-task Databricks job that orchestrates three notebooks. One notebook fails intermittently. The engineer needs to **collect detailed logs** for the failing attempts to share with the platform team. Which steps should they follow?

- A. Export notebook run results to HTML and share the file
- B. Use the notebook interactive debugger to re-run the entire multi-task job
- C. Download worker logs from the Spark UI and inspect driver/executor logs
- **D. From the job run details page, export logs or configure log delivery, then retrieve cluster logs from the compute details page**

In [ ]:
answer = "D"
explanation = (
    "The job run details page in Databricks shows per-task run history, error messages, and "
    "allows log export. Configuring cluster log delivery (Logging tab → DBFS/cloud storage path) "
    "persists driver and executor logs across multiple runs so they are available even after the "
    "cluster terminates. This is the complete and reliable approach for intermittent failures. "
    "HTML notebook export lacks Spark-level error traces. "
    "The interactive debugger is for single-notebook sessions, not multi-task job reruns. "
    "Spark UI logs are ephemeral and disappear when the cluster terminates."
)
print(f"Q15 → {answer}  |  {explanation}")